[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begelb/latent_dynamics/blob/paper/notebooks/01_leslie_2d_contraction.ipynb)

In [ ]:
# On Colab, install the CMGDB fork (a prebuilt wheel; not on PyPI)
# and the paper package. Running locally uses the project venv as is.
import sys

if "google.colab" in sys.modules:
    !pip install -q cmgdb==1.3.3+fork.2 --find-links https://github.com/bernardorivas/CMGDB/releases/expanded_assets/v1.3.3+fork.2
    !pip install -q git+https://github.com/begelb/latent_dynamics.git@paper

# Section 5.1 - Two-dimensional Leslie with contracting dynamics

## What this notebook shows

A *warm-up* for the method (paper section 5.1). We take a two-dimensional
nonlinear **Leslie population map** -- which has a period-doubling cascade, an
attracting invariant circle, and the Newhouse phenomenon, so its dynamics
resist rigorous characterization even when the map is known -- and embed it in a
**ten-dimensional** phase space with strongly contracting dynamics in the eight
extra coordinates.

From a deliberately **sparse** dataset we learn an autoencoder together with a
latent dynamics map $g$ forming an $\epsilon$-approximate semiconjugacy to the
full system, then run CMGDB on $g$ and read off the global dynamics as a Morse
graph. It has **two minimal nodes -- bistability** -- recovering both attractors
of the planar map: a stable invariant circle (Conley index $(x-1, x-1, 0)$) and
a stable period-six orbit (Conley index $(x^6-1, 0, 0)$).

### How to run

Edit the **parameters** cell below, then *Run All*. Three modes:

| `MODE` | what it does | cost |
|--------|--------------|------|
| `"replay"` | re-render the paper's saved Morse graph and Morse sets | seconds |
| `"morse"` | recompute the Morse graph of the *saved* model at your `SUBDIV` | seconds-minutes |
| `"retrain"` | run the whole pipeline from scratch with your `OVERRIDES` | minutes-hours |

**Toy subdivisions are a qualitative preview.** Coarse CMGDB grids can merge
nearby recurrent sets and change the Morse graph; the paper figures use the
config's (finer) values. The paper value for this example is noted in the
parameters cell.

In [ ]:
# ===== PARAMETERS  (edit, then Run All) ====================================
MODE = "replay"            # "replay" | "morse" | "retrain"
SEED = None                # None -> the config's default seed
SUBDIV = (10, 14, 20)      # MODE="morse": (subdiv_init, subdiv_min, subdiv_max)
                           # paper value: (27, 29, 30) -- max=30 resolves the period-6 orbit
OVERRIDES = {}             # MODE="retrain": config overrides (pydantic-validated), e.g.
                           #   {"training": {"epochs": 300}, "cmgdb": {"subdiv_max": 20}}
BOX_SCALE = "auto"         # Morse-set box size: "auto" | float | {label: float}
# ===========================================================================

In [ ]:
from latentdynamics.replay import load_experiment, retrain

REPLAY_CONFIG  = "leslie_2gen_contraction_replay"
RETRAIN_CONFIG = "leslie_2gen_contraction"

if MODE == "replay":
    exp = load_experiment(REPLAY_CONFIG, seed=SEED)
elif MODE == "morse":
    exp = load_experiment(REPLAY_CONFIG, seed=SEED).recompute_morse(subdiv=SUBDIV)
elif MODE == "retrain":
    exp = retrain(RETRAIN_CONFIG, seed=SEED, overrides=OVERRIDES)
else:
    raise ValueError(f"unknown MODE {MODE!r}")
exp

## Morse graph

The Hasse diagram of the latent dynamics. Two minimal nodes (sinks) means a bistable latent model.

In [ ]:
exp.show_morse_graph()

## Morse sets

The latent phase-space regions, colored to match the Morse graph above.

In [ ]:
exp.show_morse_sets(box_scale=BOX_SCALE)

## Run provenance

Final training losses, the CMGDB parameters used, and saved paper metrics.

In [ ]:
exp.diagnostics()